In [2]:
import psycopg2
import json
from decimal import Decimal
from datetime import datetime, date, time
from uuid import UUID


class PostgresDataEncoder(json.JSONEncoder):
    """
    Custom JSON encoder to handle PostgreSQL native types.
    Converts datetime, date, time, UUID, Decimal into JSON-serializable types.
    """
    def default(self, obj):
        if isinstance(obj, (datetime, date, time)):
            return obj.isoformat()
        elif isinstance(obj, Decimal):
            return float(obj)
        elif isinstance(obj, UUID):
            return str(obj)
        return super().default(obj)


def fetch_data_from_postgres(query, db_config):
    """
    Fetches data from PostgreSQL and returns it as a JSON string.
    Native Python types are preserved (no conversion to string).
    """
    conn = None
    result_data = []

    try:
        # Connect to PostgreSQL
        conn = psycopg2.connect(**db_config)
        cur = conn.cursor()

        # Execute query
        print(f"\n🔎 Executing query:\n{query}\n")
        cur.execute(query)
        rows = cur.fetchall()

        # Fetch column names
        col_names = [desc[0] for desc in cur.description]

        # Convert rows to list of dictionaries
        for row in rows:
            result_data.append(dict(zip(col_names, row)))

    except psycopg2.Error as e:
        print(f"❌ Error: {e}")
        return json.dumps({"error": str(e)}, indent=4)

    finally:
        if conn:
            conn.close()

    # Use the custom encoder here ✅
    return json.dumps(result_data, indent=4, cls=PostgresDataEncoder)


# ✅ PostgreSQL connection config
db_config = {
    "host": "localhost",
    "database": "postgres",
    "user": "postgres",
    "password": "root",
    "port": "5432"
}

# 📥 Get the query from user
query = input("\n🛠️ Enter your SQL query: ")

# 📤 Fetch and display the result in JSON
json_result = fetch_data_from_postgres(query, db_config)
print("\n📊 Query Result in JSON format:")
print(json_result)



🔎 Executing query:
SELECT machineid, SUM(pieces_produced_life) AS total_pieces FROM machine_data GROUP BY machineid ORDER BY total_pieces DESC;


📊 Query Result in JSON format:
[
    {
        "machineid": "CNC004",
        "total_pieces": 379577860
    },
    {
        "machineid": "CNC003",
        "total_pieces": 3375990
    }
]


In [13]:
# import pandas as pd
# import numpy as np
# import json

# def generate_chart_json(json_result: str, chart_type: str = "bar", value_column: str = None) -> dict:
#     """
#     Converts JSON result from a DataFrame into a chart configuration.

#     Args:
#         json_result (str): A JSON-formatted string representing tabular data.
#         chart_type (str): The type of chart to generate ("bar", "line", "pie").
#         value_column (str, optional): Specific column to use for pie chart values (only required for "pie").

#     Returns:
#         dict: A dictionary containing Chart.js configuration with labels and datasets.
#     """
#     # Parse JSON string to a Python dictionary, then load into a DataFrame
#     data = json.loads(json_result)
#     df = pd.DataFrame(data)
    
#     # Normalize column names to lowercase
#     df.columns = [col.lower() for col in df.columns]

#     # Attempt to infer and convert datatypes
#     for col in df.columns:
#         series = df[col]
#         inferred = pd.api.types.infer_dtype(series, skipna=True)

#         if inferred in ['datetime', 'datetime64']:
#             df[col] = pd.to_datetime(series, errors='coerce')
#         elif inferred == 'boolean':
#             df[col] = series.astype('boolean')
#         elif inferred == 'integer':
#             # Check if the integers are timestamps in milliseconds
#             if series.max() > 1e12:
#                 df[col] = pd.to_datetime(series, unit='ms', utc=True, errors='coerce')
#             else:
#                 df[col] = pd.to_numeric(series, errors='coerce').astype('Int64')
#         elif inferred == 'floating':
#             df[col] = pd.to_numeric(series, errors='coerce', downcast='float')
#         else:
#             # Try converting to datetime first, then numeric; otherwise treat as string
#             try:
#                 df[col] = pd.to_datetime(series, utc=True, errors='coerce')
#                 if df[col].notna().sum() > 0:
#                     continue
#             except:
#                 pass
#             try:
#                 df[col] = pd.to_numeric(series, errors='coerce')
#             except:
#                 df[col] = series.astype('string')

#     # Identify the label column: Prefer datetime, then string/categorical
#     label_col = None
#     for col in df.columns:
#         if pd.api.types.is_datetime64_any_dtype(df[col]):
#             label_col = col
#             break
#     if not label_col:
#         for col in df.columns:
#             if df[col].dtype == 'string' or df[col].dtype.name == 'category':
#                 label_col = col
#                 break
#     if not label_col:
#         raise ValueError("No suitable label column (datetime/string) found.")

#     # Format labels appropriately
#     labels = df[label_col].dt.strftime('%Y-%m-%d') if pd.api.types.is_datetime64_any_dtype(df[label_col]) else df[label_col].astype(str)

#     datasets = []

#     # ========== PIE CHART LOGIC ==========
#     if chart_type.lower() == "pie":
#         # Use user-specified column or fallback to first numeric column
#         if value_column is None:
#             num_cols = [col for col in df.columns if col != label_col and pd.api.types.is_numeric_dtype(df[col])]
#             if not num_cols:
#                 raise ValueError("No numeric column found for pie chart.")
#             value_column = num_cols[0]

#         # Prepare pie dataset
#         data_values = df[value_column].fillna(0).tolist()

#         # Standard color palette (repeated if more slices exist)
#         color_palette = [
#             "#FF6384", "#36A2EB", "#FFCE56", "#FF9F40",
#             "#4BC0C0", "#9966FF", "#FF6384", "#36A2EB",
#             "#FFCE56", "#FF9F40", "#4BC0C0", "#9966FF"
#         ]

#         datasets = [{
#             "label": value_column.replace("_", " ").title(),
#             "data": data_values,
#             "backgroundColor": color_palette[:len(data_values)]
#         }]

#         # Final JSON for pie chart
#         chart_json = {
#             "chart": {
#                 "type": chart_type,
#                 "data": {
#                     "labels": labels.tolist(),
#                     "datasets": datasets
#                 }
#             }
#         }

#     # ========== BAR / LINE / OTHER CHART LOGIC ==========
#     else:
#         # Use all numeric columns except the label column
#         for col in df.columns:
#             if col == label_col:
#                 continue
#             if pd.api.types.is_numeric_dtype(df[col]):
#                 datasets.append({
#                     "label": col.replace("_", " ").title(),
#                     "data": df[col].fillna(0).tolist(),
#                     "backgroundColor": "rgba(75, 192, 192, 0.2)",
#                     "borderColor": "rgba(75, 192, 192, 1)",
#                     "borderWidth": 1
#                 })

#         # Final JSON for bar/line chart
#         chart_json = {
#             "chart": {
#                 "type": chart_type,
#                 "data": {
#                     "labels": labels.tolist(),
#                     "datasets": datasets
#                 },
#                 "options": {
#                     "scales": {
#                         "yAxes": [{
#                             "ticks": {
#                                 "beginAtZero": True
#                             }
#                         }]
#                     }
#                 }
#             }
#         }

#     return chart_json
# generate_chart_json(json_result, chart_type = "bar")

In [14]:
import datetime

json_data = [
    {'machineid': 'CNC003', 'date': datetime.date(2025, 3, 10), 'total_parts_produced': 179},
    {'machineid': 'CNC004', 'date': datetime.date(2025, 3, 10), 'total_parts_produced': 99},
    {'machineid': 'CNC003', 'date': datetime.date(2025, 3, 11), 'total_parts_produced': 313},
    {'machineid': 'CNC004', 'date': datetime.date(2025, 3, 11), 'total_parts_produced': 223},
    {'machineid': 'CNC003', 'date': datetime.date(2025, 3, 12), 'total_parts_produced': 203},
    {'machineid': 'CNC004', 'date': datetime.date(2025, 3, 12), 'total_parts_produced': 341},
    {'machineid': 'CNC003', 'date': datetime.date(2025, 3, 13), 'total_parts_produced': 95},
    {'machineid': 'CNC004', 'date': datetime.date(2025, 3, 13), 'total_parts_produced': 162},
    {'machineid': 'CNC003', 'date': datetime.date(2025, 3, 14), 'total_parts_produced': 0},
    {'machineid': 'CNC004', 'date': datetime.date(2025, 3, 14), 'total_parts_produced': 0},
    {'machineid': 'CNC003', 'date': datetime.date(2025, 3, 15), 'total_parts_produced': 286},
    {'machineid': 'CNC004', 'date': datetime.date(2025, 3, 15), 'total_parts_produced': 337},
    {'machineid': 'CNC003', 'date': datetime.date(2025, 3, 16), 'total_parts_produced': 268},
    {'machineid': 'CNC004', 'date': datetime.date(2025, 3, 16), 'total_parts_produced': 203},
    {'machineid': 'CNC003', 'date': datetime.date(2025, 3, 17), 'total_parts_produced': 439},
    {'machineid': 'CNC004', 'date': datetime.date(2025, 3, 17), 'total_parts_produced': 518},
    {'machineid': 'CNC003', 'date': datetime.date(2025, 3, 18), 'total_parts_produced': 134},
    {'machineid': 'CNC004', 'date': datetime.date(2025, 3, 18), 'total_parts_produced': 360},
    {'machineid': 'CNC003', 'date': datetime.date(2025, 3, 19), 'total_parts_produced': 105},
    {'machineid': 'CNC004', 'date': datetime.date(2025, 3, 19), 'total_parts_produced': 572},
    {'machineid': 'CNC003', 'date': datetime.date(2025, 3, 20), 'total_parts_produced': 0},
    {'machineid': 'CNC004', 'date': datetime.date(2025, 3, 20), 'total_parts_produced': 283},
    {'machineid': 'CNC003', 'date': datetime.date(2025, 3, 21), 'total_parts_produced': 92},
    {'machineid': 'CNC004', 'date': datetime.date(2025, 3, 21), 'total_parts_produced': 515},
    {'machineid': 'CNC003', 'date': datetime.date(2025, 3, 22), 'total_parts_produced': 0},
    {'machineid': 'CNC004', 'date': datetime.date(2025, 3, 22), 'total_parts_produced': 0},
    {'machineid': 'CNC003', 'date': datetime.date(2025, 3, 23), 'total_parts_produced': 0},
    {'machineid': 'CNC004', 'date': datetime.date(2025, 3, 23), 'total_parts_produced': 0},
    {'machineid': 'CNC003', 'date': datetime.date(2025, 3, 24), 'total_parts_produced': 108},
    {'machineid': 'CNC004', 'date': datetime.date(2025, 3, 24), 'total_parts_produced': 227},
    {'machineid': 'CNC003', 'date': datetime.date(2025, 3, 25), 'total_parts_produced': 357},
    {'machineid': 'CNC004', 'date': datetime.date(2025, 3, 25), 'total_parts_produced': 296},
    {'machineid': 'CNC003', 'date': datetime.date(2025, 3, 26), 'total_parts_produced': 114},
    {'machineid': 'CNC004', 'date': datetime.date(2025, 3, 26), 'total_parts_produced': 100},
    {'machineid': 'CNC003', 'date': datetime.date(2025, 3, 27), 'total_parts_produced': 219},
    {'machineid': 'CNC004', 'date': datetime.date(2025, 3, 27), 'total_parts_produced': 189},
    {'machineid': 'CNC003', 'date': datetime.date(2025, 3, 28), 'total_parts_produced': 303},
    {'machineid': 'CNC004', 'date': datetime.date(2025, 3, 28), 'total_parts_produced': 226},
    {'machineid': 'CNC003', 'date': datetime.date(2025, 3, 29), 'total_parts_produced': 246},
    {'machineid': 'CNC004', 'date': datetime.date(2025, 3, 29), 'total_parts_produced': 215},
    {'machineid': 'CNC003', 'date': datetime.date(2025, 3, 30), 'total_parts_produced': 111},
    {'machineid': 'CNC004', 'date': datetime.date(2025, 3, 30), 'total_parts_produced': 104},
    {'machineid': 'CNC003', 'date': datetime.date(2025, 3, 31), 'total_parts_produced': 99},
    {'machineid': 'CNC004', 'date': datetime.date(2025, 3, 31), 'total_parts_produced': 84},
    {'machineid': 'CNC003', 'date': datetime.date(2025, 4, 1), 'total_parts_produced': 55},
    {'machineid': 'CNC004', 'date': datetime.date(2025, 4, 1), 'total_parts_produced': 39},
    {'machineid': 'CNC003', 'date': datetime.date(2025, 4, 2), 'total_parts_produced': 157},
    {'machineid': 'CNC004', 'date': datetime.date(2025, 4, 2), 'total_parts_produced': 116},
    {'machineid': 'CNC003', 'date': datetime.date(2025, 4, 3), 'total_parts_produced': 60},
    {'machineid': 'CNC004', 'date': datetime.date(2025, 4, 3), 'total_parts_produced': 42},
    {'machineid': 'CNC003', 'date': datetime.date(2025, 4, 4), 'total_parts_produced': 0},
    {'machineid': 'CNC004', 'date': datetime.date(2025, 4, 4), 'total_parts_produced': 0},
    {'machineid': 'CNC003', 'date': datetime.date(2025, 4, 5), 'total_parts_produced': 0},
    {'machineid': 'CNC004', 'date': datetime.date(2025, 4, 5), 'total_parts_produced': 129},
    {'machineid': 'CNC003', 'date': datetime.date(2025, 4, 6), 'total_parts_produced': 137},
    {'machineid': 'CNC004', 'date': datetime.date(2025, 4, 6), 'total_parts_produced': 197},
    {'machineid': 'CNC003', 'date': datetime.date(2025, 4, 7), 'total_parts_produced': 156},
    {'machineid': 'CNC004', 'date': datetime.date(2025, 4, 7), 'total_parts_produced': 186},
    {'machineid': 'CNC003', 'date': datetime.date(2025, 4, 8), 'total_parts_produced': 121},
    {'machineid': 'CNC004', 'date': datetime.date(2025, 4, 8), 'total_parts_produced': 175},
    {'machineid': 'CNC003', 'date': datetime.date(2025, 4, 9), 'total_parts_produced': 0},
    {'machineid': 'CNC004', 'date': datetime.date(2025, 4, 9), 'total_parts_produced': 0}
]


In [ ]:
import json
import pandas as pd
import numpy as np
from pandas.api.types import infer_dtype

def generate_chart_json(json_result, chart_type="bar", value_column=None, label_column=None):
    """
    Converts a JSON dataset to a compatible JSON format.

    Args:
        json_result (str or list): Input data in JSON format (list of dicts or JSON string).
        chart_type (str): Type of chart to generate. Options: 'bar', 'line', 'pie'.
        value_column (str): Column name to use for chart values. If not provided, the first numeric column is used.
        label_column (str): Column name to use for labels. If not provided, the first datetime or string column is used.

    Returns:
        dict: JSON structure compatible with Chart configuration.

    Raises:
        ValueError: If no suitable label or value column is found.
    """
    
    # Load data into DataFrame
    if isinstance(json_result, str):
        data = json.loads(json_result)
    else:
        data = json_result
    df = pd.DataFrame(data)

    # Detect and convert column data types
    for col in df.columns:
        series = df[col]
        inferred = infer_dtype(series, skipna=True)

        if inferred in ["string", "unicode"]:
            df[col] = series.astype(str)
        elif inferred == "boolean":
            df[col] = series.astype(bool)
        elif inferred == "integer":
            max_len = series.dropna().astype(str).str.len().max()
            if max_len is not None and max_len >= 13:  # Likely a timestamp in milliseconds
                # Convert to datetime if it's a timestamp
                # Assume UNIX timestamp in milliseconds
                try:
                    df[col] = pd.to_datetime(series, unit="ms")
                    continue
                except Exception:
                    pass
            df[col] = pd.to_numeric(series, errors="coerce").astype("Int64")
        elif inferred == "floating":
            df[col] = pd.to_numeric(series, errors="coerce")
        else:
            # Try datetime conversion, fallback to string
            try:
                df[col] = pd.to_datetime(series)
            except Exception:
                df[col] = series.astype(str)
    print(f"inferred data types: {df.dtypes}")
    # Detect label column if not provided
    # Identify the label column: Prefer datetime, then string/object
    label_col = None

    # Check for a datetime column
    for col in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            label_col = col
            break

    # If no datetime column is found, check for string or object columns
    if not label_col:
        for col in df.columns:
            if pd.api.types.is_string_dtype(df[col]) or pd.api.types.is_object_dtype(df[col]):
                label_col = col
                break
    print(f"label column: {label_col}")

    # Raise an error if no suitable label column is found
    if not label_col:
        raise ValueError("No suitable label column (datetime/string/object) found.")
    label_column = label_col

    # Detect value column if not provided
    if not value_column:
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        if len(numeric_cols) == 0:
            raise ValueError("No suitable value column (numeric) found.")
        value_column = numeric_cols[0]

    # Extract labels
    labels = (
        df[label_column].dt.strftime("%Y-%m-%d")
        if pd.api.types.is_datetime64_any_dtype(df[label_column])
        else df[label_column].astype(str)
    )

    # Extract values
    values = df[value_column].fillna(0).tolist()

    # Build chart JSON
    chart_json = {
        "chart": {
            "type": chart_type,
            "data": {
                "labels": labels.tolist(),
                "datasets": []
            },
            "options": {
                "scales": {
                    "yAxes": [{
                        "ticks": {"beginAtZero": True}
                    }]
                }
            }
        }
    }

    if chart_type == "pie":
        # Pie chart requires single dataset with background colors
        chart_json["chart"]["data"]["datasets"].append({
            "label": value_column,
            "data": values,
            "backgroundColor": [
                "#FF6384", "#36A2EB", "#FFCE56", "#FF9F40",
                "#4BC0C0", "#9966FF", "#FF6384", "#36A2EB",
                "#FFCE56", "#4BC0C0", "#9966FF", "#FF9F40"
            ][:len(values)]
        })
    else:
        # Bar or Line chart
        chart_json["chart"]["data"]["datasets"].append({
            "label": value_column.replace("_", " ").title(),
            "data": values,
            "backgroundColor": "rgba(75, 192, 192, 0.2)",
            "borderColor": "rgba(75, 192, 192, 1)",
            "borderWidth": 1
        })

    return chart_json
# print(chart_json.to_string(indent=4))


In [11]:
generate_chart_json(json_data, chart_type = "bar")

inferred data types: machineid                       object
date                    datetime64[ns]
total_parts_produced             Int64
dtype: object
label column: date


{'chart': {'type': 'bar',
  'data': {'labels': ['2025-03-10',
    '2025-03-10',
    '2025-03-11',
    '2025-03-11',
    '2025-03-12',
    '2025-03-12',
    '2025-03-13',
    '2025-03-13',
    '2025-03-14',
    '2025-03-14',
    '2025-03-15',
    '2025-03-15',
    '2025-03-16',
    '2025-03-16',
    '2025-03-17',
    '2025-03-17',
    '2025-03-18',
    '2025-03-18',
    '2025-03-19',
    '2025-03-19',
    '2025-03-20',
    '2025-03-20',
    '2025-03-21',
    '2025-03-21',
    '2025-03-22',
    '2025-03-22',
    '2025-03-23',
    '2025-03-23',
    '2025-03-24',
    '2025-03-24',
    '2025-03-25',
    '2025-03-25',
    '2025-03-26',
    '2025-03-26',
    '2025-03-27',
    '2025-03-27',
    '2025-03-28',
    '2025-03-28',
    '2025-03-29',
    '2025-03-29',
    '2025-03-30',
    '2025-03-30',
    '2025-03-31',
    '2025-03-31',
    '2025-04-01',
    '2025-04-01',
    '2025-04-02',
    '2025-04-02',
    '2025-04-03',
    '2025-04-03',
    '2025-04-04',
    '2025-04-04',
    '2025-04-05',
  

In [4]:
import json
import pandas as pd
import numpy as np
from pandas.api.types import infer_dtype

def generate_chart_json(json_result, chart_type="line", value_column=None, label_column=None, category_column=None):
    """
    Converts a JSON dataset to a multi-series chart JSON format (especially useful for line charts).

    Args:
        json_result (str or list): Input data in JSON format (list of dicts or JSON string).
        chart_type (str): Type of chart to generate. Options: 'bar', 'line', 'pie'.
        value_column (str): Column name to use for chart values.
        label_column (str): Column name to use for X-axis labels (e.g., date).
        category_column (str): Column to group series by (e.g., machine ID).

    Returns:
        dict: JSON structure compatible with Chart configuration.
    """

    # Load data into DataFrame
    if isinstance(json_result, str):
        data = json.loads(json_result)
    else:
        data = json_result
    df = pd.DataFrame(data)

    # Convert and infer datatypes
    for col in df.columns:
        series = df[col]
        inferred = infer_dtype(series, skipna=True)

        if inferred in ["string", "unicode"]:
            df[col] = series.astype(str)
        elif inferred == "boolean":
            df[col] = series.astype(bool)
        elif inferred == "integer":
            max_len = series.dropna().astype(str).str.len().max()
            if max_len is not None and max_len >= 13:  # Likely a timestamp in milliseconds
                # Convert to datetime if it's a timestamp
                # Assume UNIX timestamp in milliseconds
                try:
                    df[col] = pd.to_datetime(series, unit="ms")
                    continue
                except Exception:
                    pass
            df[col] = pd.to_numeric(series, errors="coerce").astype("Int64")
        elif inferred == "floating":
            df[col] = pd.to_numeric(series, errors="coerce")
        else:
            try:
                df[col] = pd.to_datetime(series)
            except Exception:
                df[col] = series.astype(str)
    print(df.dtypes)

    # Auto-detect label (x-axis) column
        # Detect label column if not provided
    # Identify the label column: Prefer datetime, then string/object
    label_column = None

    # Check for a datetime column
    for col in df.columns:
        if pd.api.types.is_datetime64_any_dtype(df[col]):
            label_column = col
            break

    # If no datetime column is found, check for string or object columns
    if not label_column:
        for col in df.columns:
            if pd.api.types.is_string_dtype(df[col]) or pd.api.types.is_object_dtype(df[col]):
                label_column = col
                break

    # Raise an error if no suitable label column is found
    if not label_column:
        raise ValueError("No suitable label column (datetime/string/object) found.")
    print(f"label_column: {label_column}")

    # Auto-detect value column
    if not value_column:
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        if not numeric_cols:
            raise ValueError("No numeric value column found")
        value_column = numeric_cols[0]
        
    print(f"value_column: {value_column}")

    # Auto-detect category column (used for multiple lines or bars)
    if not category_column:
        string_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()

        # Remove label_column from string_cols if present
        if label_column in string_cols:
            string_cols.remove(label_column)

        if string_cols:
            category_column = string_cols[0]
        elif label_column != value_column:
            category_column = label_column  # Fallback to label if distinct from value
        else:
            category_column = None  # No category needed for single-series charts

    

    print(f"category_column: {category_column}")


    # Convert label column to formatted string
    df[label_column] = (
        df[label_column].dt.strftime("%Y-%m-%d")
        if pd.api.types.is_datetime64_any_dtype(df[label_column])
        else df[label_column].astype(str)
    )
    # Color palette
    color_palette = [
        "rgb(75, 192, 192)", "rgb(255, 99, 132)", "rgb(255, 206, 86)",
        "rgb(54, 162, 235)", "rgb(153, 102, 255)", "rgb(255, 159, 64)"
    ]
    if chart_type == "pie":
        # For pie chart, we expect one label column and one value column
        if category_column:
            pie_df = df.groupby(category_column)[value_column].sum().reset_index()
            labels = pie_df[category_column].astype(str).tolist()
            values = pie_df[value_column].tolist()
        else:
            labels = df[label_column].astype(str).tolist()
            values = df[value_column].tolist()

        chart_json = {
            "chart": {
                "type": "pie",
                "data": {
                    "labels": labels,
                    "datasets": [{
                        "data": values,
                        "backgroundColor": color_palette[:len(values)]
                    }]
                }
            }
        }
        return chart_json

    if chart_type in ["line", "bar"] and category_column:
        pivot_df = df.pivot(index=label_column, columns=category_column, values=value_column).fillna(0)
        pivot_df = pivot_df.sort_index()
        labels = pivot_df.index.tolist()

        datasets = []
        for i, column in enumerate(pivot_df.columns):
            datasets.append({
                "label": column,
                "data": pivot_df[column].tolist(),
                "fill": False,
                "borderColor": color_palette[i % len(color_palette)],
                "tension": 0.1
            })

        chart_json = {
            "chart": {
                "type": chart_type,
                "data": {
                    "labels": labels,
                    "datasets": datasets
                }
            }
        }
        return chart_json
    else:
        # Fallback: single-series line/bar chart
        labels = df[label_column].tolist()
        values = df[value_column].tolist()
        chart_json = {
            "chart": {
                "type": chart_type,
                "data": {
                    "labels": labels,
                    "datasets": [{
                        "label": value_column,
                        "data": values,
                        "backgroundColor": color_palette[0],
                        "borderColor": color_palette[0],
                        "fill": False,
                        "tension": 0.1
                    }]
                }
            }
        }
        return chart_json
chart_json = generate_chart_json(json_result, chart_type="bar")
chart_json


machineid       object
total_pieces     Int64
dtype: object
label_column: machineid
value_column: total_pieces
category_column: machineid


ValueError: The name machineid occurs multiple times, use a level number

In [70]:
generate_chart_json(json_result, chart_type = "pie")

machineid      object
usage_count     Int64
dtype: object
label_column: machineid
value_column: usage_count
category_column: machineid


KeyError: None

In [1]:
import json
import pandas as pd
import numpy as np
from pandas.api.types import infer_dtype

def generate_chart_json(json_result, chart_type="line", value_column=None, label_column=None, category_column=None):
    """
    Converts a JSON dataset to a multi-series chart JSON format (especially useful for line charts).

    Args:
        json_result (str or list): Input data in JSON format (list of dicts or JSON string).
        chart_type (str): Type of chart to generate. Options: 'bar', 'line', 'pie'.
        value_column (str): Column name to use for chart values.
        label_column (str): Column name to use for X-axis labels (e.g., date).
        category_column (str): Column to group series by (e.g., machine ID).

    Returns:
        dict: JSON structure compatible with Chart configuration.
    """

    # Load data into DataFrame
    if isinstance(json_result, str):
        data = json.loads(json_result)
    else:
        data = json_result
    df = pd.DataFrame(data)

    # Convert and infer datatypes
    for col in df.columns:
        series = df[col]
        inferred = infer_dtype(series, skipna=True)

        if inferred in ["string", "unicode"]:
            df[col] = series.astype(str)
        elif inferred == "boolean":
            df[col] = series.astype(bool)
        elif inferred == "integer":
            max_len = series.dropna().astype(str).str.len().max()
            if max_len is not None and max_len >= 13:  # Likely a timestamp in milliseconds
                try:
                    df[col] = pd.to_datetime(series, unit="ms")
                    continue
                except Exception:
                    pass
            df[col] = pd.to_numeric(series, errors="coerce").astype("Int64")
        elif inferred == "floating":
            df[col] = pd.to_numeric(series, errors="coerce")
        else:
            try:
                df[col] = pd.to_datetime(series)
            except Exception:
                df[col] = series.astype(str)

    # Auto-detect label column if not provided
    if label_column is None:
        label_column = None
        # Check for datetime columns first
        for col in df.columns:
            if pd.api.types.is_datetime64_any_dtype(df[col]):
                label_column = col
                break
        # If not found, check for string/object columns
        if label_column is None:
            for col in df.columns:
                if pd.api.types.is_string_dtype(df[col]) or pd.api.types.is_object_dtype(df[col]):
                    label_column = col
                    break
        # Raise error if no suitable label column found
        if label_column is None:
            raise ValueError("No suitable label column (datetime/string/object) found.")
    # Validate label column exists
    if label_column not in df.columns:
        raise ValueError(f"label_column '{label_column}' not found in data.")

    # Auto-detect value column if not provided
    if value_column is None:
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        if not numeric_cols:
            raise ValueError("No numeric value column found.")
        value_column = numeric_cols[0]
    # Validate value column is numeric
    if not pd.api.types.is_numeric_dtype(df[value_column]):
        raise ValueError(f"value_column '{value_column}' is not numeric.")

    # Auto-detect category column if not provided
    if category_column is None:
        # Find string columns excluding label_column
        string_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
        if label_column in string_cols:
            string_cols.remove(label_column)
        if string_cols:
            category_column = string_cols[0]
        else:
            # Fallback to label_column if different from value_column
            if label_column != value_column:
                category_column = label_column
            else:
                category_column = None  # Single series chart

    # Convert label column to formatted string if datetime
    if pd.api.types.is_datetime64_any_dtype(df[label_column]):
        df[label_column] = df[label_column].dt.strftime("%Y-%m-%d")
    else:
        df[label_column] = df[label_column].astype(str)

    # Color palette
    color_palette = [
        "rgb(75, 192, 192)", "rgb(255, 99, 132)", "rgb(255, 206, 86)",
        "rgb(54, 162, 235)", "rgb(153, 102, 255)", "rgb(255, 159, 64)"
    ]

    # Generate chart JSON based on type
    if chart_type == "pie":
        if category_column:
            pie_df = df.groupby(category_column)[value_column].sum().reset_index()
            labels = pie_df[category_column].astype(str).tolist()
            values = pie_df[value_column].tolist()
        else:
            labels = df[label_column].astype(str).tolist()
            values = df[value_column].tolist()

        chart_json = {
            "chart": {
                "type": "pie",
                "data": {
                    "labels": labels,
                    "datasets": [{
                        "data": values,
                        "backgroundColor": color_palette[:len(values)]
                    }]
                }
            }
        }
        return chart_json

    elif chart_type in ["line", "bar"]:
        if category_column:
            # Use pivot_table to handle duplicates by summing values
            pivot_df = df.pivot_table(
                index=label_column,
                columns=category_column,
                values=value_column,
                aggfunc="sum"
            ).fillna(0)
            pivot_df = pivot_df.sort_index()
            labels = pivot_df.index.tolist()

            datasets = []
            for i, column in enumerate(pivot_df.columns):
                datasets.append({
                    "label": str(column),
                    "data": pivot_df[column].tolist(),
                    "fill": False,
                    "borderColor": color_palette[i % len(color_palette)],
                    "tension": 0.1
                })

            chart_json = {
                "chart": {
                    "type": chart_type,
                    "data": {
                        "labels": labels,
                        "datasets": datasets
                    }
                }
            }
            return chart_json
        else:
            # Single series chart
            labels = df[label_column].tolist()
            values = df[value_column].tolist()
            chart_json = {
                "chart": {
                    "type": chart_type,
                    "data": {
                        "labels": labels,
                        "datasets": [{
                            "label": value_column,
                            "data": values,
                            "backgroundColor": color_palette[0],
                            "borderColor": color_palette[0],
                            "fill": False,
                            "tension": 0.1
                        }]
                    }
                }
            }
            return chart_json
    else:
        raise ValueError(f"Unsupported chart_type: {chart_type}. Choose from 'line', 'bar', 'pie'.")
chart_json = generate_chart_json(json_result, chart_type = "bar")
print(json.dumps(chart_json, indent=4))

NameError: name 'json_result' is not defined

In [ ]:
import plotly.graph_objects as go

def plot_from_chart_json(chart_json):
    try:
        labels = chart_json["chart"]["data"]["labels"]
        datasets = chart_json["chart"]["data"]["datasets"]
    except (KeyError, TypeError) as e:
        print(f"Invalid input structure: {e}")
        return

    # Prompt for plot type
    plot_type = input("Enter plot type (bar/line/pie): ").strip().lower()

    # Create Plotly Figure
    fig = go.Figure()

    if plot_type == "bar":
        for dataset in datasets:
            fig.add_trace(go.Bar(
                x=labels,
                y=dataset.get("data", []),
                name=dataset.get("label", "Unnamed")
            ))
        fig.update_layout(barmode='stack')

    elif plot_type == "line":
        for dataset in datasets:
            fig.add_trace(go.Scatter(
                x=labels,
                y=dataset.get("data", []),
                mode='lines+markers',
                name=dataset.get("label", "Unnamed")
            ))

    elif plot_type == "pie":
        if len(datasets) > 1:
            print("Pie chart supports only one dataset. Using the first one.")
        dataset = datasets[0]
        fig = go.Figure(go.Pie(
            labels=chart_json["chart"]["data"]["labels"],
            values=dataset.get("data", []),
            name=dataset.get("label", "Unnamed"),
            cols = dataset.get("backgroundColor", [])
        ))
    else:
        print(f"Plot type '{plot_type}' is not supported.")
        return

    # Set titles conditionally
    fig.update_layout(
        title=chart_json.get("chart", {}).get("title", {}).get("text", "Chart Visualization"),
        xaxis_title='Labels' if plot_type != "pie" else None,
        yaxis_title='Values' if plot_type != "pie" else None
    )

    fig.show()
plot_from_chart_json(chart_json)

ValueError: Invalid property specified for object of type plotly.graph_objs.Pie: 'cols'

Did you mean "hole"?

    Valid properties:
        automargin
            Determines whether outside text labels can push the
            margins.
        customdata
            Assigns extra data each datum. This may be useful when
            listening to hover, click and selection events. Note
            that, "scatter" traces also appends customdata items in
            the markers DOM elements
        customdatasrc
            Sets the source reference on Chart Studio Cloud for
            `customdata`.
        direction
            Specifies the direction at which succeeding sectors
            follow one another.
        dlabel
            Sets the label step. See `label0` for more info.
        domain
            :class:`plotly.graph_objects.pie.Domain` instance or
            dict with compatible properties
        hole
            Sets the fraction of the radius to cut out of the pie.
 

In [123]:
chart_json["chart"]["data"]["datasets"]["backgroundColor"][0]

TypeError: list indices must be integers or slices, not str